# Artistic Telephone: generate, choose, pass

This notebook has one small play area for **rounds, prompts, attempts and export**. You only need this notebook file.

[Open in Colab](https://colab.research.google.com/github/digitalideation/prompt_h2601/blob/simplify-artistic-telephone-stegg/notebooks/03_artistic_telephone.ipynb) or open it in JupyterHub. Use the [shared SWITCHdrive folder](https://drive.switch.ch/index.php/s/6tapYANksgszZez) to receive and pass images. Each generation uses your own Replicate account and may incur a charge.

## 1. Install

Run once. We use [Replicate](https://replicate.com/) for generation, [ST3GG](https://github.com/elder-plinius/ST3GG) to embed the prompt inside each PNG, and `ipywidgets` for the small interface. ST3GG is also worth exploring independently.

In [ ]:
%pip install -q replicate "stegg==3.0.0" "ipywidgets>=8.1,<9"

## 2. Connect

Enter your [Replicate token](https://replicate.com/account/api-tokens) into the hidden prompt. Never paste it into a code cell.

In [ ]:
import replicate
from getpass import getpass

client = replicate.Client(api_token=getpass("Replicate API token: ").strip(), timeout=120)

## 3. Choose your group and seat

Set these once. You will select the round in the play area below.

In [ ]:
GROUP = "A"   # A, B, C, D, E or F
SEAT = 1      # 1, 2, 3, 4 or 5

if GROUP not in "ABCDEF" or len(GROUP) != 1:
    raise ValueError("Choose a group from A to F.")
if type(SEAT) is not int or not 1 <= SEAT <= 5:
    raise ValueError("Choose a seat from 1 to 5.")

MODEL = "black-forest-labs/flux-schnell"
print(f"Group {GROUP}, seat {SEAT:02d} — ready to play.")

## 4. Play

Run the next two code cells once. Then stay in the interface.

1. Choose your **round**. The notebook tells you which PNG to **open** in SWITCHdrive and which one to **pass**.
2. **Open the incoming image locally or in SWITCHdrive.** We don't need an upload control: the image is for *you* to look at, not for Replicate.
3. Write a **prompt** and click **Generate and save**. Each click makes a potentially paid request and automatically saves a numbered PNG with that exact prompt hidden using ST3GG.
4. Choose an **attempt** to preview, click **Export**, and upload the downloaded PNG to the correct SWITCHdrive group folder. Export does **not** generate again.

Keep the original PNG unchanged. No screenshots or JPEGs: that can destroy the hidden prompt. This is concealment, not encryption; please don't decode classmates' prompts before the reveal.

If generation fails or is interrupted, check the printed prediction ID in Replicate before trying again.

In [ ]:
class GenerationError(Exception):
    """A prediction finished without a usable image."""


def generate_image(inputs, client, MODEL):
    model = client.models.get(MODEL)
    if model.latest_version is None:
        raise GenerationError("This model has no available version. Check the model with the instructor.")
    version_id = model.latest_version.id

    prediction = client.predictions.create(version=version_id, input=inputs)
    print("Prediction ID:", prediction.id)
    prediction.wait()

    if prediction.status != "succeeded":
        raise GenerationError(
            f"Prediction {prediction.id} ended with status '{prediction.status}'. "
            "Check its details in Replicate before trying again."
        )
    urls = prediction.output
    if not isinstance(urls, list) or not urls or not isinstance(urls[0], str) or not urls[0].startswith("https://"):
        raise GenerationError(
            f"Prediction {prediction.id} returned no usable image URL. "
            "Check the model's output format with the instructor."
        )
    return urls

In [ ]:
# Run once to open the game
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
import shutil
import ipywidgets as widgets
from PIL import Image as PILImage
from steg_core import create_config, encode_text, decode_text
from IPython.display import display, FileLink, clear_output

steg = create_config(channels="RGB", bits=1, compress=True, strategy="interleaved")
folder = Path("attempts") / f"Group_{GROUP}_S{SEAT:02d}"
folder.mkdir(parents=True, exist_ok=True)


def filenames(round_number):
    seats = [((SEAT - round_number + i) % 5) + 1 for i in range(round_number)]
    parts = [f"{seat:02d}" for seat in seats]
    return ("_".join(parts[:-1]) + ".png" if round_number > 1 else None,
            "_".join(parts) + ".png")


round_box = widgets.Dropdown(options=[1, 2, 3, 4, 5], description="Round:")
directions = widgets.HTML()
prompt_box = widgets.Textarea(placeholder="Describe the image you see...",
                              layout=widgets.Layout(width="100%", height="125px"))
generate_button = widgets.Button(description="Generate and save", button_style="primary")
attempt_box = widgets.Dropdown(options=[], description="Attempt:")
preview = widgets.Image(format="png", width=440)
export_button = widgets.Button(description="Export selected PNG", button_style="success")
status = widgets.HTML()
log = widgets.Output()
download = widgets.Output()


def show_selected(change=None):
    preview.value = Path(attempt_box.value).read_bytes() if attempt_box.value else b""


def refresh_attempts():
    _, outgoing = filenames(round_box.value)
    images = sorted(folder.glob(f"{outgoing[:-4]}_try*.png"))
    attempt_box.options = [(path.name, str(path)) for path in images]
    attempt_box.disabled = not images
    export_button.disabled = not images
    show_selected()


def change_round(change=None):
    incoming, outgoing = filenames(round_box.value)
    directions.value = (
        f"<b>Open:</b> {incoming or 'No incoming image — invent a scene'} "
        f"from Group_{GROUP} in SWITCHdrive"
        f"<p><b>Pass:</b> {outgoing} to seat {SEAT % 5 + 1:02d}</p>"
    )
    if change:
        prompt_box.value = ""
    refresh_attempts()
    status.value = ""


def generate_clicked(button):
    prompt_used = prompt_box.value.strip()
    if not prompt_used:
        status.value = "<b>Write a prompt first.</b>"
        return

    generate_button.disabled = True
    export_button.disabled = True
    _, outgoing = filenames(round_box.value)
    with log:
        clear_output(wait=True)
        try:
            print("Generating one image (Replicate may charge for this)...")
            urls = generate_image(
                {"prompt": prompt_used, "num_outputs": 1,
                 "aspect_ratio": "1:1", "output_format": "png"},
                client, MODEL,
            )
            with urlopen(urls[0], timeout=120) as response:
                original = PILImage.open(BytesIO(response.read()))
                encoded = encode_text(original, prompt_used, steg)
            if decode_text(encoded, steg) != prompt_used:
                raise ValueError("Could not verify the hidden prompt.")

            number = 1
            while (folder / f"{outgoing[:-4]}_try{number:02d}.png").exists():
                number += 1
            saved = folder / f"{outgoing[:-4]}_try{number:02d}.png"
            encoded.save(saved, format="PNG")
            refresh_attempts()
            attempt_box.value = str(saved)
            status.value = f"<b>Saved:</b> {saved.name}"
        except Exception as error:
            print(f"Could not finish ({type(error).__name__}).")
            print("Check the prediction in Replicate before clicking Generate again.")
            status.value = "<b>Generation or saving failed.</b>"
        finally:
            generate_button.disabled = False
            export_button.disabled = not bool(attempt_box.options)


def export_clicked(button):
    if not attempt_box.value:
        return
    _, outgoing = filenames(round_box.value)
    destination = Path(f"Group_{GROUP}") / outgoing
    destination.parent.mkdir(exist_ok=True)
    shutil.copyfile(attempt_box.value, destination)
    with download:
        clear_output(wait=True)
        print("Upload this file to SWITCHdrive:", destination)
        display(FileLink(str(destination)))
        try:
            from google.colab import files
        except ImportError:
            pass
        else:
            files.download(str(destination))


round_box.observe(change_round, names="value")
attempt_box.observe(show_selected, names="value")
generate_button.on_click(generate_clicked)
export_button.on_click(export_clicked)
change_round()

display(widgets.VBox([
    round_box, directions,
    widgets.HTML("<b>Your prompt</b>"), prompt_box, generate_button,
    status, log,
    widgets.HTML("<b>Choose an attempt</b>"), attempt_box,
    preview, export_button, download,
], layout=widgets.Layout(width="95%", max_width="680px")))

**Next round:** change the round dropdown, write the new prompt and repeat. Your group and seat stay fixed. Keep only the chosen original PNG for each round in SWITCHdrive.

[ST3GG](https://github.com/elder-plinius/ST3GG) is used here for prompt hiding; you can also explore it for other steganography experiments. Remember that Colab's local attempt files disappear when the runtime is deleted, so download your own attempts if you want to keep them.